In [7]:
# Basic packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Model-related
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings('ignore')


In [11]:
# Load Dataset
df=pd.read_excel(r"C:\Users\ak32a\Downloads\Anal-ysts\LTF Challenge data with dictionary.xlsx", sheet_name="TrainData")

In [9]:
# View structure
print("Data shape:", df.shape)
df.head()

Data shape: (47970, 105)


,FarmerID,State,REGION,SEX,CITY,Zipcode,DISTRICT,VILLAGE,MARITAL_STATUS,Location,...,Rabi Seasons Agro Ecological Sub Zone in 2020,Rabi Seasons Seasonal average groundwater thickness (cm) in 2020,Rabi Seasons Seasonal average groundwater replenishment rate (cm) in 2020,Night light index,Village score based on socio-economic parameters (Non normalised),Village score based on socio-economic parameters (0 to 100),"Village category based on socio-economic parameters (Good, Average, Poor)",Land Holding Index source (Total Agri Area/ no of people),Road density (Km/ SqKm),Target_Variable/Total Income
0,1002818465057450,MADHYA PRADESH,CENTRAL,M,BARELI,464668,RAISEN,Seoni,M,NaN,...,CENTRAL HIGHLANDS (MALWA AND BUNDELKHAND) HOT...,97.24,19.50,0.95,22.380262,33.527178,Poor,0.773129,0.00,1360000
1,1012300674433870,BIHAR,EAST,M,BANDRA,848125,MUZAFFARPUR,Namapur,M,NaN,...,DECCAN PLATEAU (TELANGANA) AND EASTERN GHATS ...,73.96,16.76,0.97,24.630262,37.173626,Poor,0.454140,0.00,807200
2,1013472263587380,MADHYA PRADESH,CENTRAL,M,MALHARGARH,458556,MANDSAUR,Billaud,M,NaN,...,CENTRAL HIGHLANDS ( MALWA ) GUJARAT PLAIN AND...,90.05,22.44,0.95,19.493313,28.848462,Poor,0.657040,0.00,500000
3,1019525480704050,MAHARASHTRA,WEST,M,RENAPUR,413527,LATUR,Renapur,M,NaN,...,DECCAN PLATU HOT SEMI-ARID ECO-REGION,94.64,21.48,0.98,31.836367,48.852156,Average,0.235615,2.49,558000
4,1021915867444260,MADHYA PRADESH,CENTRAL,F,KHURAI,470117,SAGAR,Singhpur,M,NaN,...,CENTRAL HIGHLANDS (MALWA AND BUNDELKHAND) HOT...,95.90,18.93,0.97,21.327371,31.820817,Poor,0.207264,0.00,800000


In [12]:
def split_min_max(temp_series):
    return temp_series.str.split('/', expand=True).astype(float)

# Identify all columns with temp ranges (those with "min & max" in their names)
temp_cols = [col for col in df.columns if "min & max" in col]

for col in temp_cols:
    new_cols = [f"{col}_min", f"{col}_max"]
    df[new_cols] = split_min_max(df[col].astype(str))
    df.drop(columns=[col], inplace=True)


In [15]:
# Handle 'Good', 'Average', 'Poor' as ordinal
grade_map = {'Poor': 0, 'Average': 1, 'Good': 2}

for col in df.columns:
    if df[col].nunique() <= 3 and df[col].dropna().isin(grade_map.keys()).all():
        df[col] = df[col].map(grade_map)

# Label encode remaining object columns
for col in df.select_dtypes(include='object').columns:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))


In [21]:
# Mandi Coordinates

In [31]:
mandi_names = df['K022-Nearest Mandi Name'].dropna().unique()


In [33]:
!pip install geopy


In [ ]:
from geopy.geocoders import Nominatim
import time

geolocator = Nominatim(user_agent="mandi_locator")

def get_coords(mandi_name):
    try:
        location = geolocator.geocode(f"{mandi_name}, India", timeout=10)
        if location:
            return location.latitude, location.longitude
    except:
        return None, None
    return None, None

# Create a mapping dictionary
mandi_coord_map = {}
for mandi in mandi_names:
    lat, lon = get_coords(mandi)
    mandi_coord_map[mandi] = (lat, lon)
    time.sleep(1)  # to respect API rate limit


In [ ]:
df[['Mandi_Lat', 'Mandi_Lon']] = df['K022-Nearest Mandi Name'].map(mandi_coord_map).apply(pd.Series)


In [ ]:
from geopy.distance import geodesic

def calc_distance(row):
    if pd.notnull(row['Latitude']) and pd.notnull(row['Mandi_Lat']):
        return geodesic((row['Latitude'], row['Longitude']), (row['Mandi_Lat'], row['Mandi_Lon'])).km
    return np.nan

df['distance_to_mandi'] = df.apply(calc_distance, axis=1)
